In [ ]:
# Example 1: Basic Function Decorators and Wrapping
import wrapt
import time
import functools

# Basic decorator using wrapt
@wrapt.decorator
def timing_decorator(wrapped, instance, args, kwargs):
    """Decorator to measure function execution time"""
    start_time = time.time()
    try:
        result = wrapped(*args, **kwargs)
        return result
    finally:
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"Function '{wrapped.__name__}' took {execution_time:.4f} seconds")

# Apply decorator to functions
@timing_decorator
def slow_function(n):
    """A function that takes some time to execute"""
    time.sleep(0.1)
    return sum(range(n))

@timing_decorator
def fast_function(x, y):
    """A fast function for comparison"""
    return x + y

print("Basic Function Decorators and Wrapping:")

# Test the decorated functions
result1 = slow_function(1000)
print(f"slow_function(1000) = {result1}")

result2 = fast_function(5, 10)
print(f"fast_function(5, 10) = {result2}")

# Demonstrate proper introspection
print(f"\nIntrospection preserved:")
print(f"slow_function.__name__: {slow_function.__name__}")
print(f"slow_function.__doc__: {slow_function.__doc__}")
print(f"slow_function.__module__: {slow_function.__module__}")

# Compare with functools.wraps
def traditional_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Traditional wrapper calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@traditional_decorator
def traditional_wrapped():
    """Traditional wrapped function"""
    return "traditional"

print(f"\nComparison with traditional decorator:")
print(f"Traditional: {type(traditional_wrapped)}")
print(f"Wrapt: {type(slow_function)}")

print("Basic function decorators and wrapping complete")

In [ ]:
# Example 2: Advanced Decorator Patterns and Method Wrapping
import wrapt
import inspect
from datetime import datetime

# Parameterized decorator
def logging_decorator(prefix="LOG", include_args=True):
    """Parameterized decorator for logging function calls"""
    @wrapt.decorator
    def wrapper(wrapped, instance, args, kwargs):
        # Build call information
        call_info = f"[{prefix}] {wrapped.__name__}"
        
        if include_args and (args or kwargs):
            arg_strs = [repr(arg) for arg in args]
            kwarg_strs = [f"{k}={repr(v)}" for k, v in kwargs.items()]
            all_args = arg_strs + kwarg_strs
            call_info += f"({', '.join(all_args)})"
        
        print(f"{datetime.now().strftime('%H:%M:%S')} - {call_info}")
        
        result = wrapped(*args, **kwargs)
        print(f"{datetime.now().strftime('%H:%M:%S')} - {call_info} -> {repr(result)}")
        return result
    
    return wrapper

# Method wrapping in classes
class Calculator:
    """Example class with decorated methods"""
    
    def __init__(self, name):
        self.name = name
        self.operations_count = 0
    
    @logging_decorator(prefix="CALC", include_args=True)
    def add(self, a, b):
        """Add two numbers"""
        self.operations_count += 1
        return a + b
    
    @logging_decorator(prefix="CALC", include_args=False)
    def multiply(self, a, b):
        """Multiply two numbers"""
        self.operations_count += 1
        return a * b
    
    @wrapt.decorator
    def count_operations(wrapped, instance, args, kwargs):
        """Decorator to count method calls"""
        if instance:
            instance.operations_count += 1
        return wrapped(*args, **kwargs)
    
    @count_operations
    def divide(self, a, b):
        """Divide two numbers"""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

# Dynamic method wrapping after class creation
class ManagedProperty:
    """Example of wrapping methods dynamically"""
    
    def __init__(self):
        self._value = 0
        self._access_count = 0
    
    def get_value(self):
        """Get the current value"""
        self._access_count += 1
        return self._value
    
    def set_value(self, new_value):
        """Set a new value"""
        self._value = new_value

# Wrapper for dynamic method wrapping
def wrap_method_with_logging(instance, method_name):
    """Wrap an instance method with logging"""
    original_method = getattr(instance, method_name)
    
    @wrapt.decorator
    def logging_wrapper(wrapped, inst, args, kwargs):
        print(f"Calling {method_name} on {inst.__class__.__name__}")
        result = wrapped(*args, **kwargs)
        print(f"Result: {repr(result)}")
        return result
    
    # Create wrapped version and bind it
    wrapped_method = logging_wrapper(original_method)
    setattr(instance, method_name, wrapped_method)

print("Advanced Decorator Patterns and Method Wrapping:")

# Test calculator with method decoration
calc = Calculator("MyCalculator")
print(f"\nTesting {calc.name}:")

result1 = calc.add(5, 3)
result2 = calc.multiply(4, 6)
result3 = calc.divide(10, 2)

print(f"\nOperations completed: {calc.operations_count}")

# Test dynamic method wrapping
print("\nTesting dynamic method wrapping:")
prop_obj = ManagedProperty()

# Wrap the methods dynamically
wrap_method_with_logging(prop_obj, 'get_value')
wrap_method_with_logging(prop_obj, 'set_value')

# Use the wrapped methods
prop_obj.set_value(42)
retrieved_value = prop_obj.get_value()
print(f"Retrieved value: {retrieved_value}")
print(f"Access count: {prop_obj._access_count}")

# Demonstrate signature preservation
print(f"\nSignature preservation:")
print(f"calc.add signature: {inspect.signature(calc.add)}")
print(f"calc.multiply signature: {inspect.signature(calc.multiply)}")

print("Advanced decorator patterns and method wrapping complete")

In [ ]:
# Example 3: Object Proxies and Transparent Wrappers
import wrapt

# Simple transparent object proxy
class SimpleProxy(wrapt.ObjectProxy):
    """Basic proxy that passes through all calls"""
    
    def __init__(self, wrapped):
        super(SimpleProxy, self).__init__(wrapped)
    
    def custom_method(self):
        """Custom method added by the proxy"""
        return f"Proxy wrapping {self.__wrapped__.__class__.__name__}"

# Counting proxy
class CountingProxy(wrapt.ObjectProxy):
    """Proxy that counts method calls"""
    pass

# Example class to wrap
class DataContainer:
    """Simple data container for demonstration"""
    
    def __init__(self, name):
        self.name = name
        self.data = {}
        self.count = 0
    
    def add_item(self, key, value):
        """Add an item to the container"""
        self.data[key] = value
        self.count += 1
        return f"Added {key}: {value}"
    
    def get_item(self, key):
        """Get an item from the container"""
        return self.data.get(key)
    
    def get_count(self):
        """Get item count"""
        return self.count
    
    def __str__(self):
        return f"DataContainer({self.name}, {self.count} items)"
    
    def __len__(self):
        return self.count

# Function-based proxy creation
def create_function_proxy(func):
    """Create a proxy wrapper for a function"""
    
    @wrapt.decorator
    def wrapper(wrapped, instance, args, kwargs):
        print(f"Calling {wrapped.__name__} with args={args}, kwargs={kwargs}")
        result = wrapped(*args, **kwargs)
        print(f"Result: {repr(result)}")
        return result
    
    return wrapper(func)

print("Object Proxies and Transparent Wrappers:")

# Test simple proxy
original_container = DataContainer("OriginalContainer")
proxied_container = SimpleProxy(original_container)

print(f"\nOriginal type: {type(original_container)}")
print(f"Proxied type: {type(proxied_container)}")
print(f"Proxied isinstance check: {isinstance(proxied_container, DataContainer)}")

# Use the proxied object - it behaves exactly like the original
print(f"\nUsing proxied container:")
print(f"Name: {proxied_container.name}")
result1 = proxied_container.add_item("key1", "value1")
print(f"Add result: {result1}")

proxied_container.add_item("key2", "value2")
retrieved = proxied_container.get_item("key1")
print(f"Retrieved: {retrieved}")

print(f"Container string: {str(proxied_container)}")
print(f"Container length: {len(proxied_container)}")

# Access custom proxy method
print(f"Custom proxy method: {proxied_container.custom_method()}")

# Verify proxy transparency
print(f"\nProxy transparency:")
print(f"proxied_container.count: {proxied_container.count}")
print(f"proxied_container.get_count(): {proxied_container.get_count()}")

# Test that modifications work through proxy
proxied_container.name = "ModifiedName"
print(f"Modified name through proxy: {proxied_container.name}")

# Verify the original is modified (proxy is transparent)
print(f"Original object name: {original_container.name}")

# Test function wrapping
print(f"\nFunction wrapping:")

def calculate(x, y, operation="add"):
    """Sample calculation function"""
    if operation == "add":
        return x + y
    elif operation == "multiply":
        return x * y
    return None

wrapped_calculate = create_function_proxy(calculate)

result1 = wrapped_calculate(5, 3)
result2 = wrapped_calculate(4, 6, operation="multiply")

# Verify function metadata is preserved
print(f"\nFunction metadata preserved:")
print(f"Function name: {wrapped_calculate.__name__}")
print(f"Function doc: {wrapped_calculate.__doc__}")

# Demonstrate ObjectProxy with built-in types
print(f"\nProxy with built-in types:")
list_proxy = wrapt.ObjectProxy([1, 2, 3, 4, 5])
print(f"Original list: {list_proxy}")
list_proxy.append(6)
print(f"After append: {list_proxy}")
print(f"Length: {len(list_proxy)}")
print(f"Slicing: {list_proxy[2:4]}")

dict_proxy = wrapt.ObjectProxy({"a": 1, "b": 2})
print(f"\nOriginal dict: {dict_proxy}")
dict_proxy["c"] = 3
print(f"After adding key: {dict_proxy}")
print(f"Keys: {list(dict_proxy.keys())}")

print("Object proxies and transparent wrappers complete")

In [ ]:
# Example 4: Monkey Patching and Dynamic Wrapping
import wrapt
import sys
import types

# Safe monkey patching with wrapt
def patch_method(target_class, method_name, wrapper_func):
    """Safely patch a method with proper restoration"""
    original_method = getattr(target_class, method_name)
    
    @wrapt.decorator
    def method_wrapper(wrapped, instance, args, kwargs):
        return wrapper_func(wrapped, instance, args, kwargs)
    
    wrapped_method = method_wrapper(original_method)
    setattr(target_class, method_name, wrapped_method)
    
    return original_method  # Return for potential restoration

# Example class to patch
class Calculator:
    """Calculator class for monkey patching demonstration"""
    
    def __init__(self):
        self.operations = []
    
    def add(self, a, b):
        """Add two numbers"""
        return a + b
    
    def subtract(self, a, b):
        """Subtract two numbers"""
        return a - b
    
    def multiply(self, a, b):
        """Multiply two numbers"""
        return a * b

# Wrapper functions for patching
def operation_logger(wrapped, instance, args, kwargs):
    """Log mathematical operations"""
    operation_name = wrapped.__name__
    if len(args) >= 2:
        a, b = args[0], args[1]
        print(f"Performing {operation_name}: {a} {operation_name} {b}")
    
    result = wrapped(*args, **kwargs)
    
    if instance and hasattr(instance, 'operations'):
        instance.operations.append({
            'operation': operation_name,
            'args': args,
            'result': result
        })
    
    print(f"Result: {result}")
    return result

def validation_wrapper(wrapped, instance, args, kwargs):
    """Validate inputs for mathematical operations"""
    # Check for numeric inputs
    for i, arg in enumerate(args):
        if not isinstance(arg, (int, float)):
            raise TypeError(f"Argument {i+1} must be numeric, got {type(arg).__name__}")
    
    # Special validation for division-like operations
    if wrapped.__name__ in ['divide', 'subtract'] and len(args) >= 2 and args[1] == 0:
        if wrapped.__name__ == 'divide':
            raise ValueError("Cannot divide by zero")
    
    return wrapped(*args, **kwargs)

# Dynamic wrapper application
def apply_wrapper_to_methods(target_class, method_names, wrapper_func):
    """Apply wrapper to multiple methods dynamically"""
    original_methods = {}
    
    for method_name in method_names:
        if hasattr(target_class, method_name):
            original_methods[method_name] = patch_method(target_class, method_name, wrapper_func)
            print(f"Patched {target_class.__name__}.{method_name}")
        else:
            print(f"Method {method_name} not found in {target_class.__name__}")
    
    return original_methods

# Context manager for temporary patching
class TemporaryPatch:
    """Context manager for temporary method patching"""
    
    def __init__(self, target_class, method_name, wrapper_func):
        self.target_class = target_class
        self.method_name = method_name
        self.wrapper_func = wrapper_func
        self.original_method = None
    
    def __enter__(self):
        self.original_method = patch_method(self.target_class, self.method_name, self.wrapper_func)
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # Restore original method
        setattr(self.target_class, self.method_name, self.original_method)

print("Monkey Patching and Dynamic Wrapping:")

# Test original calculator
calc = Calculator()
print(f"\nOriginal calculator:")
print(f"5 + 3 = {calc.add(5, 3)}")
print(f"Operations logged: {len(calc.operations)}")

# Apply logging patch
print(f"\nApplying logging patch:")
original_methods = apply_wrapper_to_methods(
    Calculator, 
    ['add', 'subtract', 'multiply'], 
    operation_logger
)

# Test patched calculator
calc2 = Calculator()
print(f"\nPatched calculator:")
calc2.add(10, 5)
calc2.subtract(20, 8)
calc2.multiply(4, 7)

print(f"\nOperations logged: {len(calc2.operations)}")
for op in calc2.operations:
    print(f"  {op['operation']}: {op['args']} -> {op['result']}")

# Test temporary patching
print(f"\nTesting temporary patch with validation:")

with TemporaryPatch(Calculator, 'add', validation_wrapper):
    calc3 = Calculator()
    print(f"Valid operation: {calc3.add(1, 2)}")
    
    try:
        calc3.add("invalid", 2)  # Should raise TypeError
    except TypeError as e:
        print(f"Validation caught error: {e}")

# After context, original behavior restored
print(f"\nAfter temporary patch - original behavior:")
calc4 = Calculator()
result = calc4.add(1, 2)  # No validation
print(f"Operation without validation: {result}")

print("Monkey patching and dynamic wrapping complete")

In [ ]:
# Example 5: Production Decorator Framework and Performance Monitoring
import wrapt
import time
import threading
import functools
from collections import defaultdict, deque
from datetime import datetime, timedelta
import statistics

class PerformanceMonitor:
    """Production-ready performance monitoring system using wrapt"""
    
    def __init__(self, max_history=1000):
        self.metrics = defaultdict(lambda: {
            'call_count': 0,
            'total_time': 0.0,
            'min_time': float('inf'),
            'max_time': 0.0,
            'recent_times': deque(maxlen=100),
            'error_count': 0,
            'last_called': None
        })
        self.lock = threading.Lock()
        self.enabled = True
    
    def record_call(self, func_name, execution_time, error=False):
        """Record function call metrics"""
        if not self.enabled:
            return
        
        with self.lock:
            metrics = self.metrics[func_name]
            
            if error:
                metrics['error_count'] += 1
            else:
                metrics['call_count'] += 1
                metrics['total_time'] += execution_time
                metrics['min_time'] = min(metrics['min_time'], execution_time)
                metrics['max_time'] = max(metrics['max_time'], execution_time)
                metrics['recent_times'].append(execution_time)
            
            metrics['last_called'] = datetime.now()
    
    def get_stats(self, func_name=None):
        """Get performance statistics"""
        with self.lock:
            if func_name:
                if func_name in self.metrics:
                    metrics = self.metrics[func_name]
                    return self._calculate_stats(func_name, metrics)
                return None
            else:
                return {name: self._calculate_stats(name, metrics) 
                       for name, metrics in self.metrics.items()}
    
    def _calculate_stats(self, func_name, metrics):
        """Calculate comprehensive statistics"""
        if metrics['call_count'] == 0:
            return {
                'function': func_name,
                'call_count': 0,
                'error_count': metrics['error_count'],
                'avg_time': 0,
                'last_called': metrics['last_called']
            }
        
        recent_times = list(metrics['recent_times'])
        
        stats = {
            'function': func_name,
            'call_count': metrics['call_count'],
            'error_count': metrics['error_count'],
            'total_time': metrics['total_time'],
            'avg_time': metrics['total_time'] / metrics['call_count'],
            'min_time': metrics['min_time'],
            'max_time': metrics['max_time'],
            'last_called': metrics['last_called']
        }
        
        if recent_times:
            stats['recent_avg'] = statistics.mean(recent_times)
            if len(recent_times) > 1:
                stats['recent_stdev'] = statistics.stdev(recent_times)
        
        return stats

# Global monitor instance
monitor = PerformanceMonitor()

# Production decorator factory
def performance_monitor(include_args=False, threshold_warning=1.0):
    """Decorator factory for performance monitoring"""
    
    @wrapt.decorator
    def wrapper(wrapped, instance, args, kwargs):
        func_name = f"{wrapped.__module__}.{wrapped.__qualname__}" if hasattr(wrapped, '__qualname__') else wrapped.__name__
        
        start_time = time.time()
        error_occurred = False
        
        try:
            result = wrapped(*args, **kwargs)
            return result
        except Exception as e:
            error_occurred = True
            raise
        finally:
            execution_time = time.time() - start_time
            monitor.record_call(func_name, execution_time, error=error_occurred)
            
            # Warning for slow operations
            if execution_time > threshold_warning:
                print(f"⚠️  Slow operation detected: {func_name} took {execution_time:.3f}s")
    
    return wrapper

# Caching decorator with performance monitoring
def cached_with_monitoring(maxsize=128):
    """LRU cache with performance monitoring"""
    
    def decorator(func):
        # Apply LRU cache first
        cached_func = functools.lru_cache(maxsize=maxsize)(func)
        
        # Then apply performance monitoring
        @performance_monitor()
        @wrapt.decorator
        def wrapper(wrapped, instance, args, kwargs):
            result = wrapped(*args, **kwargs)
            
            # Log cache statistics periodically
            cache_info = wrapped.cache_info()
            if cache_info.hits + cache_info.misses > 0:
                hit_rate = cache_info.hits / (cache_info.hits + cache_info.misses)
                if (cache_info.hits + cache_info.misses) % 50 == 0:  # Log every 50 calls
                    print(f"Cache stats for {func.__name__}: {hit_rate:.1%} hit rate, {cache_info.currsize}/{cache_info.maxsize} size")
            
            return result
        
        # Wrap the cached function
        monitored_func = wrapper(cached_func)
        
        # Preserve cache methods
        monitored_func.cache_info = cached_func.cache_info
        monitored_func.cache_clear = cached_func.cache_clear
        
        return monitored_func
    
    return decorator

# Example functions for testing
@performance_monitor(threshold_warning=0.05)
def cpu_intensive_task(n):
    """Simulate CPU-intensive work"""
    result = 0
    for i in range(n):
        result += i ** 2
    return result

@performance_monitor()
def io_simulation(delay=0.01):
    """Simulate I/O operation"""
    time.sleep(delay)
    return f"IO completed after {delay}s"

@cached_with_monitoring(maxsize=64)
def expensive_calculation(x, y):
    """Expensive calculation with caching"""
    time.sleep(0.01)  # Simulate expensive work
    return x ** y + y ** x

@performance_monitor()
def error_prone_function(value):
    """Function that might raise errors"""
    if value < 0:
        raise ValueError("Negative values not allowed")
    return value * 2

# Demonstrate production monitoring
print("Production Decorator Framework and Performance Monitoring:")

# Run various operations
print("\nExecuting monitored operations...")

# CPU-intensive tasks
for i in range(3):
    result = cpu_intensive_task(10000 + i * 5000)

# I/O simulations
for delay in [0.01, 0.02, 0.03]:
    io_simulation(delay)

# Cached operations
for i in range(10):
    # Some cache hits, some misses
    x, y = (i % 3) + 1, (i % 2) + 1
    expensive_calculation(x, y)

# Error testing
for value in [5, 10, -1, 15]:
    try:
        error_prone_function(value)
    except ValueError:
        pass  # Expected for negative values

# Generate performance report
print("\nPerformance Report:")
print("=" * 60)

all_stats = monitor.get_stats()
for func_name, stats in all_stats.items():
    print(f"\nFunction: {stats['function']}")
    print(f"  Calls: {stats['call_count']}, Errors: {stats['error_count']}")
    print(f"  Avg time: {stats['avg_time']*1000:.2f}ms")
    print(f"  Min/Max: {stats['min_time']*1000:.2f}ms / {stats['max_time']*1000:.2f}ms")
    
    if 'recent_avg' in stats:
        print(f"  Recent avg: {stats['recent_avg']*1000:.2f}ms")
    
    if stats['last_called']:
        time_since = datetime.now() - stats['last_called']
        print(f"  Last called: {time_since.total_seconds():.1f}s ago")

# Cache statistics
print(f"\nCache Statistics:")
cache_info = expensive_calculation.cache_info()
hit_rate = cache_info.hits / (cache_info.hits + cache_info.misses) * 100
print(f"Cache hit rate: {hit_rate:.1f}%")
print(f"Cache size: {cache_info.currsize}/{cache_info.maxsize}")

print("Production decorator framework and performance monitoring complete")